# Reproducibility: Benchmark Figures from Snapshot CSVs

This notebook loads the pre-computed snapshot CSVs from `data/results/snapshots/paper_v1/`
and reproduces the key figures and tables from the paper — no re-computation required.

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path

## 2. Load snapshot CSVs

In [ ]:
SNAPSHOT_DIR = Path("../data/results/snapshots/paper_v1")

df_unit = pd.read_csv(SNAPSHOT_DIR / "results_per_unit.csv")
df_method = pd.read_csv(SNAPSHOT_DIR / "results_per_method.csv")
df_dataset = pd.read_csv(SNAPSHOT_DIR / "results_per_dataset.csv")

print(f"Per-unit rows:    {len(df_unit):,}")
print(f"Per-method rows:   {len(df_method)}")
print(f"Per-dataset rows:  {len(df_dataset)}")

### Quick peek at the data

In [ ]:
df_unit.head(3)

In [ ]:
df_method

In [ ]:
print("\nColumns in per-unit:", list(df_unit.columns))
print("Columns in per-method:", list(df_method.columns))
print("Columns in per-dataset:", list(df_dataset.columns))

## 3. Method-by-Dataset Heatmap of Mean AP

We use `results_per_dataset.csv` which already contains the pre-aggregated `mean_ap` per (method, dataset) pair.

In [ ]:
# Build a method x dataset pivot table from the pre-aggregated results
heatmap_data = df_dataset.pivot_table(
    index="method_id", columns="dataset_id", values="mean_ap", aggfunc="first"
)

# Sort methods by their overall mean AP (descending)
method_order = df_method.sort_values("mean_ap", ascending=False)["method_id"].tolist()
method_order = [m for m in method_order if m in heatmap_data.index]
heatmap_data = heatmap_data.loc[method_order]

print(f"Heatmap shape: {heatmap_data.shape[0]} methods x {heatmap_data.shape[1]} datasets")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

cmap = plt.cm.viridis.copy()
cmap.set_bad("#eeeeee")  # grey for missing (e.g. scCAD / DeepScena fallback units)

im = ax.imshow(heatmap_data.values, aspect="auto", cmap=cmap, vmin=0, vmax=1)

ax.set_xticks(range(len(heatmap_data.columns)))
ax.set_xticklabels(heatmap_data.columns, rotation=45, ha="right", fontsize=9)
ax.set_yticks(range(len(heatmap_data.index)))
ax.set_yticklabels(heatmap_data.index, fontsize=10)

cbar = fig.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label("Mean Average Precision (AP)")

ax.set_title("Method-by-Dataset Heatmap — Mean AP", fontsize=14, fontweight="bold")

plt.tight_layout()
plt.show()

## 4. Prevalence Bin Bar Chart

We bin the prevalence column from `results_per_unit.csv` into discrete ranges and show the
mean AP achieved by each method within each prevalence bin.

In [ ]:
prevalence_bins = [0, 0.001, 0.01, 0.05, 0.10, 0.20, 0.50, 1.0]
prevalence_labels = [
    "<0.1%", "0.1–1%", "1–5%", "5–10%", "10–20%", "20–50%", ">50%"
]

df_unit["prevalence_bin"] = pd.cut(
    df_unit["prevalence"], bins=prevalence_bins, labels=prevalence_labels,
    include_lowest=True
)

bin_summary = (
    df_unit.groupby(["method_id", "prevalence_bin"], observed=False)["ap"]
    .agg(["mean", "count"])
    .reset_index()
)
bin_summary.head()

In [ ]:
methods_to_show = ["hvg_logreg", "CaSee", "FiRE", "expr_threshold", "scMalignantFinder",
                   "cellsius", "scCAD", "RareQ", "DeepScena", "random_baseline"]

colors = plt.cm.tab10(np.linspace(0, 1, len(methods_to_show)))

x = np.arange(len(prevalence_labels))
width = 0.75 / len(methods_to_show)

fig, ax = plt.subplots(figsize=(14, 5))

for i, method in enumerate(methods_to_show):
    subset = bin_summary[bin_summary["method_id"] == method].set_index("prevalence_bin")
    means = [subset.loc[lab, "mean"] if lab in subset.index else np.nan for lab in prevalence_labels]
    offset = (i - len(methods_to_show) / 2 + 0.5) * width
    ax.bar(x + offset, means, width, label=method, color=colors[i], edgecolor="white", linewidth=0.3)

ax.set_xticks(x)
ax.set_xticklabels(prevalence_labels, fontsize=10)
ax.set_ylabel("Mean AP")
ax.set_xlabel("Prevalence Bin")
ax.set_title("Method Performance by Prevalence Bin", fontsize=13, fontweight="bold")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8, frameon=False)
ax.set_ylim(0, 1.05)

plt.tight_layout()
plt.show()

## 5. Leaderboard Table

Aggregate method ranking from `results_per_method.csv`, sorted by mean AP.

In [ ]:
leaderboard_cols = [
    "method_id", "category", "mean_ap", "median_ap", "std_ap",
    "mean_auroc", "mean_f1_top_k", "mean_runtime_s",
    "n_units", "n_datasets", "n_fallback"
]

leaderboard = df_method[leaderboard_cols].sort_values("mean_ap", ascending=False).reset_index(drop=True)
leaderboard["rank"] = range(1, len(leaderboard) + 1)
leaderboard = leaderboard[["rank"] + leaderboard_cols]

# Format numeric columns
for col in ["mean_ap", "median_ap", "std_ap", "mean_auroc", "mean_f1_top_k"]:
    leaderboard[col] = leaderboard[col].map(lambda x: f"{x:.3f}")
leaderboard["mean_runtime_s"] = leaderboard["mean_runtime_s"].map(lambda x: f"{x:.1f}")

leaderboard

## 6. Summary

All figures and tables above are generated exclusively from the snapshot CSVs already committed
to the repository at `data/results/snapshots/paper_v1/`. No model re-training or heavy
dependencies are required — only numpy, pandas, and matplotlib.